# ESP32-S3-4848S040 — actualización y flash desde PowerShell

Este notebook conserva en Google Drive el mismo procedimiento del README del repositorio `wpv10barza/erp-mantto-esp32`.

> El notebook sirve como guía en Colab/Drive. El flash físico se ejecuta en Windows PowerShell porque el ESP32 está conectado al puerto COM local.

## Actualizar repositorio

```powershell
cd $HOME\erp-mantto-esp32
git fetch origin main --prune
git checkout main
git reset --hard origin/main
git clean -fd -e include/local_config.h

$Main = (git rev-parse HEAD).Trim()
$Remote = ((git ls-remote origin refs/heads/main) -split "\s+")[0]
"LOCAL  = $Main"
"REMOTE = $Remote"
if ($Main -ne $Remote) { throw "El repositorio local no coincide con origin/main" }
```

## Instalar PlatformIO

```powershell
py -m pip install --upgrade platformio==6.2.0
pio --version
```

## Configuración privada

```powershell
if (-not (Test-Path .\include\local_config.h)) {
  Copy-Item .\include\local_config.example.h .\include\local_config.h
  notepad .\include\local_config.h
}
git check-ignore .\include\local_config.h
```

No guardes contraseñas ni tokens dentro del notebook o GitHub.

## Pruebas

```powershell
python .\tests\test_panel_state_contract.py
python .\tests\test_final_ui_contract.py
python .\tests\test_wifi_source.py
python .\tests\test_command_editor_integration.py
python .\tests\test_touch_gpio_contract.py
python .\test\command_buffer_regression.py
python .\tests\test_production_firmware.py
pio test -e native
```

## Compilar producción

```powershell
pio run -e panel_4848s040
```

## Detectar puerto COM

```powershell
pio device list
[System.IO.Ports.SerialPort]::GetPortNames()
```

## Flash completo limpio

```powershell
$Port = "COM5"
pio run -e panel_4848s040 -t erase --upload-port $Port
pio run -e panel_4848s040 -t upload --upload-port $Port
```

## Monitor serie

```powershell
pio device monitor --port $Port --baud 115200
```

## Validación física

Comprueba:
- precisión de los botones del menú principal;
- retorno `INICIO` desde `EDITAR ORDEN 3C`;
- teclado ABC/123 centrado;
- BKSP, SPACE, ENTER;
- izquierda/derecha, DEL y LIMPIAR;
- `PROBAR CLOUD`;
- `ENVIAR 3C`.

## Ver GitHub Actions

```powershell
gh run list --repo wpv10barza/erp-mantto-esp32 --branch main --limit 10
```

No uses como versión final un commit cuyo workflow `Firmware CD` esté en failure.
